# Локальные FEM-данные: рассеяние волн

Запускайте в локальном **Python (DOLFINx Complex)**. Все данные сохраняются в `neural_operator/data/static_low_frequency_1000x8`; Google Drive и Colab не нужны. Первое выполнение создаёт данные, повторное использует каталог с той же конфигурацией.

На $\Omega=[-2,2]^2$ решаем для $u^s$:
\[
\int A\nabla u^s\cdot\nabla\bar v-k^2n u^s\bar v\,dx-ik\int_{\partial\Omega}u^s\bar v\,ds
=\int(1-A)\nabla u^i\cdot\nabla\bar v+k^2(n-1)u^i\bar v\,dx,
\quad u^i=e^{ik(x\cos\theta+y\sin\theta)}.
\]
Материалы DG0 задаются на общей конформной сетке с 24 кругами и крупными разделительными линиями. Одна LU-факторизация переиспользуется для направлений одной частоты. Узловые A,n служат входами сети; для повторного FEM/GMsFEM сохраняются исходные поэлементные коэффициенты.

**Статичная задача:** геометрия и исходная среда №0 фиксированы; A,n и DG0-материалы одинаковы во всех примерах. Меняются k и θ. Один medium_id=0, разбиение по frequency_id: 670/170/160 частот (5360/1360/1280 полей). 1000 частот × 8 направлений = 8000 полей на полосу. Этот эксперимент проверяет интерполяцию параметров волны на одной среде.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from neural_operator.settings import (BASE, N_FREQUENCIES, N_DIRECTIONS, DATASET_TAG, POD_METHOD,
    LOW_DATA_DIR, LOW_RUNS, SVD_RUNS, COMPARISON_RUNS)
DATA_DIR = LOW_DATA_DIR
RUNS = LOW_RUNS
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.datasets import load_dataset, write_dataset_info
from neural_operator.plotting import display_table, plot_prediction, plot_history


In [ ]:
from neural_operator.datasets import StaticDatasetConfig, generate_dataset
CONFIG = StaticDatasetConfig(n_frequencies=N_FREQUENCIES, n_directions=N_DIRECTIONS, h=.10, n_coarse=4, k_range=(1., 2.), seed=2026)
dataset = generate_dataset(DATA_DIR, CONFIG)
info = write_dataset_info(dataset)
print('Данные:', dataset.directory)
print('u_scat:', dataset.u_scat.shape, 'треугольников:', len(dataset.triangles))


In [ ]:
train, val, test = dataset.split_indices()
display_table([{'split': name, 'samples': len(ids), 'media': len(np.unique(dataset.medium_id[ids])), 'frequency_groups': len(np.unique(dataset.split_group_ids[ids]))}
               for name, ids in zip(['train', 'validation', 'test'], [train, val, test])])
print('Интеграл единицы:', float(np.ones(len(dataset.coordinates)) @ dataset.mass_matrix @ np.ones(len(dataset.coordinates))))


In [ ]:
import matplotlib.tri as mtri
from neural_operator.fields import scattering_components
tri = mtri.Triangulation(*dataset.coordinates.T, dataset.triangles)
i = int(train[0])
components = scattering_components(dataset.u_scat[i])
fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
for ax, values, title in zip(axes[0, :2], [dataset.A[i], dataset.n[i]],
                             ['A: nodal input', 'n: nodal input']):
    artist = ax.tripcolor(tri, values, shading='gouraud')
    ax.set(aspect='equal', title=title)
    fig.colorbar(artist, ax=ax, shrink=.8)
axes[0, 2].set_axis_off()
for ax, key, label in zip(axes[1], ['real', 'imag', 'abs'], ['Re', 'Im', 'Abs']):
    values = components[key]
    limit = max(float(np.max(np.abs(values))), 1e-12)
    cmap, lo = ('viridis', 0) if key == 'abs' else ('RdBu_r', -limit)
    artist = ax.tripcolor(tri, values, shading='gouraud', cmap=cmap, vmin=lo, vmax=limit)
    ax.set(aspect='equal', title=f'{label}(u_scattering): FEM')
    fig.colorbar(artist, ax=ax, shrink=.8)
plt.show()


## Независимое уточнение сетки

Проверяем один validation-пример с максимальным k. Это оценка изменения FEM-решения при h → h/2, включая перенос исходного P1-поля в узлы уточнённой сетки. Она отделена от ошибки сети относительно обучающих FEM-меток. Проверки однородной среды и независимой UFL-сборки находятся в `tests/test_fem_data.py`.


In [ ]:
from neural_operator.benchmark import compare_refined_fem, mesh_resolution
index = int(val[np.argmax(dataset.k0[val])])
refinement = compare_refined_fem(dataset, index)
display_table([{'sample': index, **mesh_resolution(dataset, index), **refinement}])
result_dir = BASE / 'runs' / f'dataset_{DATASET_TAG}'
result_dir.mkdir(parents=True, exist_ok=True)
(result_dir / 'validation_refinement.json').write_text(json.dumps(refinement, indent=2))


Далее: [SVD-data-checks.ipynb](SVD-data-checks.ipynb), затем три training notebooks. Для большого исследования измените CONFIG и DATA_DIR вместе; существующие данные другой конфигурации не перезаписываются. Формат старого `fem_dataset.npz` поддерживается при наличии соседнего `geometry.npz`, но сравнение PDE требует сохранённых DG0-массивов и `etal.msh`.
